In [ ]:
def histogram(mdata, odata, mlabel, olabel, title, xlabel, ylabel, n_bins=16, vrange=[-5,35], ylim=[0,0.38]):
    """
    make histogram comparing model and observation data
    """
    n_bins = n_bins
    vrange = vrange
    
    mdata_rshp = mdata.compressed()
    mweights = np.ones_like(mdata_rshp)/len(mdata_rshp)
    odata_rshp = odata.compressed()
    oweights = np.ones_like(odata_rshp)/len(odata_rshp)
    
    plt.figure(figsize=(8,4), facecolor='w')
    plt.hist(odata_rshp, n_bins, range=vrange, weights=oweights, histtype='step', stacked=True, fill=False,
            label=olabel, color='b', linewidth='2')
    plt.hist(mdata_rshp, n_bins, range=vrange, weights=mweights, histtype='step', stacked=True, fill=False,
            label=mlabel, color='r', linewidth='2')
    plt.xlabel(xlabel, size=15)
    plt.ylabel(ylabel, size=15)
    plt.xticks(size=15)
    plt.yticks(size=15)
    plt.ylim(ylim)
    plt.legend(loc=2, fontsize=15)
    plt.title(title, size=16, fontweight='bold')
    plt.show()
    
def interpolation(data1, data2, XC1, YC1, XC2, YC2, start=2):
    """
    Make interpolated array which has same grid as data2. 1을 2랑 똑같이 만들어줌
    data.shape = [nt,ny,nx]
    """
    from scipy.interpolate import interp2d
    
    mask1 = data1[0,:,:].mask   
    mask2 = data2[0,start::4,start::4].mask
    mask = np.logical_or(mask1, mask2)
    nt, ny, nx = data1[:,:,:].shape
    
    itpData2 = np.ma.empty([nt,ny,nx])
    for i in range(nt):
        g = interp2d(XC2, YC2, data2[i,:,:], kind='cubic')
        if start==2:
            temp = g(XC1, YC1)[1:-1,1:-1]
        elif start==0:
            temp = g(XC1, YC1)
        itpData2[i,:,:] = np.ma.masked_array(temp,mask)
        
    return itpData2

# Warm pool mld trend
def modelTrend(mdata, title='', startY=1991, finalY=2010, ylim=[0,100], ylabel='m', color='r'):
    
    yrmm = []
    for yr in np.arange(startY,finalY+1):
        for mm in np.arange(1,13):
            yyyymm = str(yr)+'-'+str(mm)
            yrmm.append(yyyymm)
            
    wpData = mdata[:,60:181,80:241] # before interpolate
    nt, ny, nx = wpData.shape
    
    plt.figure(figsize=(20,3))
    plt.plot(yrmm, np.ma.average(np.ma.average(wpData, axis=2), axis=1), color, label='GloSea5')
    plt.xticks(yrmm[::24], fontsize=15)
    plt.xlabel('Time', fontsize=15)
    plt.ylim(ylim)
    plt.yticks(fontsize=15)
    plt.ylabel(ylabel, fontsize=15)
    plt.ticklabel_format(style='sci', axis='y', fontsize=15)
    plt.legend(fontsize=15)
    plt.title(title, size=16, fontweight='bold')
    
def calmld(rho, depth, drho=0.03):
    """
    Estimate MLD using a density criteria on depth coordinate.
    drho[kg/m3]
    """
    [nz, ny, nx] = rho.shape
    vout = np.ma.zeros([ny, nx])
    
    # Compute MLD
    for ix in range(nx):
        for iy in range(ny):
            if np.isnan(rho[0, iy, ix])==True:
                vout[iy, ix] = np.nan
            else:
                rhocol = rho[:, iy, ix]
#                 crirho = rhocol[2] + drho # rho_10m + drho
                crirho = rhocol[0] + drho # rho_0m + drho
                ic = np.isfinite(rhocol)
                # find the pressure value where rho = rho_surface + drho
#                 mldpres = np.interp(crirho, rhocol[ic], pres[ic])
                vout[iy,ix] = np.interp(crirho, rhocol[ic], depth[ic])
    
    return vout
def heatContent(T,nt,nz,ny,nx, mld=False, mldData=None, ZC=None, z=7):
    """
    Calculate heat content in MLD or given depth. 
    """
    if mld == True:
        diff = np.ma.empty([nt,nz,ny,nx])
        mldidx = np.ma.empty([nt,nz,ny,nx])
        for iz in range(nz):
            diff[:,iz,:,:] = abs(ZC[iz] - mldData[:,:,:])
        mldidx = np.argmin(diff, axis=1)
        
        mldidx_mask = np.zeros([nt,nz,ny,nx])
        for it in range(nt):
            for iy in range(ny):
                for ix in range(nx):
                    idx = int(mldidx[it,iy,ix])
                    for iz in range(idx):
                        mldidx_mask[it,iz,iy,ix] = 1
        
    density = 1.035e3 # mean sea water density
    Cw = 3.895e3 # specific heat capacity of sea water

    if mld == True : 
        TZ = np.ma.empty([nt,nz,ny,nx])
        for iz in range(nz-1):
            TZ[:,iz,:,:] = T[:,iz,:,:]*(ZC[iz+1]-ZC[iz])
    else :
        TZ = np.ma.empty([nt,z,ny,nx])
        for iz in range(z):
            TZ[:,iz,:,:] = T[:,iz,:,:]*(ZC[iz+1]-ZC[iz])
        
    if mld == True:
        TZ = TZ*mldidx_mask # 해당 idx 까지는 남기고 나머지는 0 처리
    
    HC = np.ma.sum(TZ, axis=1) * density * Cw
    
    return HC